In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "../data/processed/feature_engineered.csv"
)

print("Dataset shape:", df.shape)
print(df.head())

C:\Users\Naveen kumar\AppData\Local\Temp\ipykernel_9408\3682199530.py:4: DtypeWarning: Columns (0: exercise_intensity) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


Dataset shape: (64283, 66)
           Patient            timestamp  glucose_value meal_type  meal_carbs  \
0  540-ws-training  2027-05-19 11:36:29             76      none         0.0   
1  540-ws-training  2027-05-19 11:41:29             72      none         0.0   
2  540-ws-training  2027-05-19 11:46:29             68      none         0.0   
3  540-ws-training  2027-05-19 11:51:29             65      none         0.0   
4  540-ws-training  2027-05-19 11:56:29             63      none         0.0   

  bolus_type  bolus_dose  basal_value  temp_basal_value exercise_type  ...  \
0     normal         0.8         0.95               0.0          none  ...   
1     normal         0.8         0.95               0.0          none  ...   
2     normal         0.8         0.95               0.0          none  ...   
3     normal         0.8         0.95               0.0          none  ...   
4     normal         0.8         0.95               0.0          none  ...   

   sleep_change gsr_rol

In [2]:
print("Target exists:", "glucose_30min_ahead" in df.columns)

Target exists: True


In [3]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

df = df.sort_values(
    ["Patient", "timestamp"]
).reset_index(drop=True)

print(df[["Patient", "timestamp", "glucose_30min_ahead"]].head())

           Patient           timestamp  glucose_30min_ahead
0  540-ws-training 2027-05-19 11:36:29                 71.0
1  540-ws-training 2027-05-19 11:41:29                 78.0
2  540-ws-training 2027-05-19 11:46:29                 90.0
3  540-ws-training 2027-05-19 11:51:29                 99.0
4  540-ws-training 2027-05-19 11:56:29                110.0


In [4]:
drop_cols = [
    "Patient",
    "timestamp",
    "glucose_30min_ahead",
    "meal_type",
    "bolus_type",
    "exercise_type",
    "exercise_intensity",
    "stressor_type",
    "stressor_description",
    "illness_type",
    "illness_description",
    "time_period"
]

X = df.drop(
    columns=drop_cols,
    errors="ignore"
)

y = df["glucose_30min_ahead"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (64283, 54)
y shape: (64283,)


In [5]:
# Make sure data is sorted
df = df.sort_values(
    ["Patient", "timestamp"]
).reset_index(drop=True)

# Create train/test masks patient-wise
train_parts = []
test_parts = []

for patient, patient_df in df.groupby("Patient"):
    
    split_index = int(len(patient_df) * 0.8)
    
    train_parts.append(patient_df.iloc[:split_index])
    test_parts.append(patient_df.iloc[split_index:])

train_df = pd.concat(train_parts).reset_index(drop=True)
test_df = pd.concat(test_parts).reset_index(drop=True)

print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)

Training shape: (51424, 66)
Testing shape: (12859, 66)


In [6]:
print("\nTraining date range:")
print(train_df["timestamp"].min())
print(train_df["timestamp"].max())

print("\nTesting date range:")
print(test_df["timestamp"].min())
print(test_df["timestamp"].max())


Training date range:
2025-04-16 11:17:05
2027-06-23 21:09:15

Testing date range:
2025-05-17 06:36:54
2027-07-03 23:26:44


In [7]:
print("\nPatient-wise split:")
for patient in df["Patient"].unique():
    
    train_patient = train_df[
        train_df["Patient"] == patient
    ]
    
    test_patient = test_df[
        test_df["Patient"] == patient
    ]
    
    print(
        patient,
        "| Train:", len(train_patient),
        "| Test:", len(test_patient)
    )


Patient-wise split:
540-ws-training | Train: 9431 | Test: 2358
544-ws-training | Train: 8392 | Test: 2099
552-ws-training | Train: 7092 | Test: 1774
567-ws-training | Train: 8430 | Test: 2108
584-ws-training | Train: 9487 | Test: 2372
596-ws-training | Train: 8592 | Test: 2148


In [8]:
target = "glucose_30min_ahead"

X_train = train_df.drop(
    columns=drop_cols,
    errors="ignore"
)

y_train = train_df[target]

X_test = test_df.drop(
    columns=drop_cols,
    errors="ignore"
)

y_test = test_df[target]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (51424, 54)
y_train: (51424,)
X_test: (12859, 54)
y_test: (12859,)


In [9]:
print("Non-numeric training columns:")

print(
    X_train.select_dtypes(
        exclude=np.number
    ).columns.tolist()
)

Non-numeric training columns:
['work_intensity']


In [10]:
print("\nNumber of features:", X_train.shape[1])
print("\nData types:")
print(X_train.dtypes)


Number of features: 54

Data types:
glucose_value                       int64
meal_carbs                        float64
bolus_dose                        float64
basal_value                       float64
temp_basal_value                  float64
exercise_duration                 float64
gsr                               float64
skin_temperature                  float64
acceleration                      float64
is_sleeping                         int64
sleep_quality                     float64
is_working                          int64
work_intensity                        str
hypo_event                          int64
illness_event                       int64
fingerstick_glucose               float64
hour                                int64
day_of_week                         int64
is_weekend                          int64
glucose_prev_5min                 float64
glucose_prev_10min                float64
glucose_prev_15min                float64
glucose_prev_30min                float

In [11]:
print("Work intensity values:")
print(X_train["work_intensity"].value_counts(dropna=False))

Work intensity values:
work_intensity
none    47448
5.0      1631
3.0      1142
2.0       690
4.0       413
6.0       100
Name: count, dtype: int64


In [12]:
X_train = pd.get_dummies(
    X_train,
    columns=["work_intensity"],
    dtype=float
)

X_test = pd.get_dummies(
    X_test,
    columns=["work_intensity"],
    dtype=float
)

# Make sure train and test have exactly the same columns
X_train, X_test = X_train.align(
    X_test,
    join="left",
    axis=1,
    fill_value=0
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\nRemaining non-numeric columns:")
print(
    X_train.select_dtypes(
        exclude=np.number
    ).columns.tolist()
)

X_train shape: (51424, 59)
X_test shape: (12859, 59)

Remaining non-numeric columns:
[]


In [13]:
print("Missing values in X_train:")
print(X_train.isna().sum().sum())

print("\nMissing values in X_test:")
print(X_test.isna().sum().sum())

Missing values in X_train:


38961

Missing values in X_test:
8716


In [14]:
X_train_np = X_train.astype("float32").values
X_test_np = X_test.astype("float32").values

y_train_np = y_train.astype("float32").values
y_test_np = y_test.astype("float32").values

print("X_train:", X_train_np.shape)
print("X_test:", X_test_np.shape)
print("y_train:", y_train_np.shape)
print("y_test:", y_test_np.shape)

X_train: (51424, 59)
X_test: (12859, 59)
y_train: (51424,)
y_test: (12859,)


In [15]:
import sys

print(sys.version)
print(sys.executable)

3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
C:\Users\Naveen kumar\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe


In [16]:
%pip install --upgrade pip
%pip install --upgrade tensorflow keras

Looking in indexes: https://pypi.org/simple/
Note: you may need to restart the kernel to use updated packages.


Looking in indexes: https://pypi.org/simple/
Note: you may need to restart the kernel to use updated packages.


In [17]:
import tensorflow as tf
import keras

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("Python:", tf.sysconfig.get_build_info())

normalizer = keras.layers.Normalization()
normalizer.adapt(X_train_np)

print("Normalization layer ready.")

TensorFlow: 2.21.0
Keras: 3.15.1
Python: OrderedDict([('is_cuda_build', False), ('is_rocm_build', False), ('is_tensorrt_build', False), ('msvcp_dll_names', 'msvcp140.dll,msvcp140_1.dll')])
Normalization layer ready.


In [18]:
model = tf.keras.Sequential([
    normalizer,

    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.2),

    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.2),

    tf.keras.layers.Dense(32, activation="relu"),

    # Regression output
    tf.keras.layers.Dense(1)
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ normalization (Normalization)   │ (51424, 59)            │           119 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 119 (480.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 119 (480.00 B)

In [19]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=[
        tf.keras.metrics.MeanAbsoluteError(name="mae")
    ]
)

print("Model compiled successfully.")

Model compiled successfully.


In [20]:
history = model.fit(
    X_train_np,
    y_train_np,
    validation_split=0.2,
    epochs=50,
    batch_size=64,
    verbose=1
)

Epoch 1/50
643/643 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 28271.7500 - mae: 156.4691 - val_loss: 26796.3398 - val_mae: 153.6576
Epoch 2/50
643/643 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 28071.4277 - mae: 155.8280 - val_loss: 26599.7598 - val_mae: 153.0162
Epoch 3/50
643/643 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 27872.2891 - mae: 155.1876 - val_loss: 26404.3496 - val_mae: 152.3768
Epoch 4/50
643/643 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 27674.2383 - mae: 154.5478 - val_loss: 26210.0098 - val_mae: 151.7376
Epoch 5/50
643/643 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 27477.1504 - mae: 153.9090 - val_loss: 26016.5625 - val_mae: 151.0989
Epoch 6/50
643/643 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 27280.9297 - mae: 153.2703 - val_loss: 25823.9980 - val_mae: 150.4603
Epoch 7/50
643/643 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 27085.6328 - mae: 152.6321 - val_loss: 25632.2461 - val_mae: 149.8216
Epoch 8/50
643/643 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 26891.2637 - mae: 151.9930 - v

In [21]:
print(history.history.keys())

dict_keys(['loss', 'mae', 'val_loss', 'val_mae'])


In [22]:
test_loss, test_mae = model.evaluate(
    X_test_np,
    y_test_np,
    verbose=1
)

print("Test MSE:", test_loss)
print("Test MAE:", test_mae)

402/402 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 20971.3594 - mae: 131.1156
Test MSE: 20971.359375
Test MAE: 131.11558532714844


In [23]:
y_pred = model.predict(X_test_np).flatten()

print("Actual values:")
print(y_test_np[:10])

print("\nPredicted values:")
print(y_pred[:10])

402/402 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
Actual values:
[176. 173. 171. 165. 158. 154. 151. 146. 143. 141.]

Predicted values:
[31.888632 31.888632 31.888632 31.888632 31.888632 31.888632 31.888632
 31.888632 31.888632 31.888632]


In [25]:
pip install -U scikit-learn


Looking in indexes: https://pypi.org/simple/
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ----- ---------------------------------- 1.0/8.3 MB 5.0 MB/s eta 0:00:02
   -------- ------------------------------- 1.8/8.3 MB 4.2 MB/s eta 0:00:02
   ------------ --------------------------- 2.6/8.3 MB 4.2 MB/s eta 0:00:02
   ---------------- ----------------------- 3.4/8.3 MB 4.1 MB/s eta 0:00:02
   -------------------- ------------------- 4.2/8.3 MB 4.1 MB/s eta 0:00:02
   ------------------------ --------------- 5.0/8.3 MB 3.9 MB/s eta 0:00:01
   ----------------------------- ---------- 6.0/8.3 MB 4.0 MB/s eta 0:00:01
   ------------------------------- -------- 6.6/8.3 MB 4.0 MB/s eta 0:00:01
   ----------------------------------- ---- 7.3/8.3 MB 3.9 MB/s eta 0:00:01
   ---------------------------------------- 8.3/8.3 MB 3.9 MB/s  

In [26]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test_np, y_pred)
rmse = np.sqrt(mean_squared_error(y_test_np, y_pred))
r2 = r2_score(y_test_np, y_pred)

print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)

MAE : 131.1149444580078
RMSE: 144.8149074573298
R²  : -4.547644138336182


In [27]:
print("Training target:")
print(y_train.describe())

print("\nTest target:")
print(y_test.describe())

print("\nPrediction:")
print(pd.Series(y_pred).describe())

Training target:
count    51424.000000
mean       156.291070
std         60.576725
min         40.000000
25%        111.000000
50%        148.000000
75%        191.000000
max        400.000000
Name: glucose_30min_ahead, dtype: float64

Test target:
count    12859.000000
mean       163.003577
std         61.485946
min         40.000000
25%        117.000000
50%        156.000000
75%        201.000000
max        400.000000
Name: glucose_30min_ahead, dtype: float64

Prediction:
count    12859.000000
mean        31.888634
std          0.000000
min         31.888632
25%         31.888632
50%         31.888632
75%         31.888632
max         31.888632
dtype: float64


In [28]:
print("First 20 actual:", y_test_np[:20])
print("First 20 predicted:", y_pred[:20])

First 20 actual: [176. 173. 171. 165. 158. 154. 151. 146. 143. 141. 139. 135. 134. 133.
 132. 127. 122. 121. 119. 118.]
First 20 predicted: [31.888632 31.888632 31.888632 31.888632 31.888632 31.888632 31.888632
 31.888632 31.888632 31.888632 31.888632 31.888632 31.888632 31.888632
 31.888632 31.888632 31.888632 31.888632 31.888632 31.888632]


In [29]:
baseline_prediction = np.full(
    len(y_test_np),
    y_train_np.mean()
)

baseline_mae = mean_absolute_error(
    y_test_np,
    baseline_prediction
)

baseline_rmse = np.sqrt(
    mean_squared_error(y_test_np, baseline_prediction)
)

baseline_r2 = r2_score(
    y_test_np,
    baseline_prediction
)

print("Baseline MAE :", baseline_mae)
print("Baseline RMSE:", baseline_rmse)
print("Baseline R²  :", baseline_r2)

Baseline MAE : 48.86790466308594
Baseline RMSE: 61.84888771925794
Baseline R²  : -0.011919260025024414


In [30]:
print("Mean actual     :", y_test_np.mean())
print("Mean prediction :", y_pred.mean())

print("Min actual      :", y_test_np.min())
print("Max actual      :", y_test_np.max())

print("Min prediction  :", y_pred.min())
print("Max prediction  :", y_pred.max())

Mean actual     : 163.00357
Mean prediction : 31.888634
Min actual      : 40.0
Max actual      : 400.0
Min prediction  : 31.888632
Max prediction  : 31.888632


In [31]:
train_pred = model.predict(X_train_np).flatten()

print("Training actual mean:", y_train_np.mean())
print("Training prediction mean:", train_pred.mean())

print("\nTraining prediction:")
print("Min :", train_pred.min())
print("Max :", train_pred.max())
print("Mean:", train_pred.mean())

1607/1607 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
Training actual mean: 156.29108
Training prediction mean: 31.888632

Training prediction:
Min : 31.888632
Max : 31.888632
Mean: 31.888632


In [32]:
print("First training loss:", history.history["loss"][0])
print("Last training loss :", history.history["loss"][-1])

print("First validation loss:", history.history["val_loss"][0])
print("Last validation loss :", history.history["val_loss"][-1])

First training loss: 28271.75
Last training loss : 19469.123046875
First validation loss: 26796.33984375
Last validation loss : 18170.0


In [33]:
print("X_train shape:", X_train_np.shape)
print("y_train shape:", y_train_np.shape)

print("\nX_train range:")
print("Min:", np.min(X_train_np))
print("Max:", np.max(X_train_np))

print("\ny_train range:")
print("Min:", np.min(y_train_np))
print("Max:", np.max(y_train_np))

X_train shape: (51424, 59)
y_train shape: (51424,)

X_train range:
Min: nan
Max: nan

y_train range:
Min: 40.0
Max: 400.0


In [34]:
print("Any NaN in X_train:", np.isnan(X_train_np).any())
print("Any Inf in X_train:", np.isinf(X_train_np).any())

print("Any NaN in y_train:", np.isnan(y_train_np).any())
print("Any Inf in y_train:", np.isinf(y_train_np).any())

Any NaN in X_train: True
Any Inf in X_train: False
Any NaN in y_train: False
Any Inf in y_train: False


In [35]:
missing = X_train.isna().sum()

missing = missing[missing > 0].sort_values(ascending=False)

print(missing)

sleep_quality                     32064
gsr_rolling_std_30min               701
skin_temp_rolling_std_30min         701
acceleration_rolling_std_30min      701
skin_temperature                    695
skin_temp_rolling_15min             695
gsr                                 695
acceleration                        695
acceleration_rolling_15min          695
gsr_rolling_15min                   695
bolus_dose                          235
fingerstick_glucose                 173
glucose_prev_30min                   36
glucose_rate_30min                   36
glucose_change_30min                 36
glucose_prev_15min                   18
glucose_change_15min                 18
glucose_rate_15min                   18
glucose_prev_10min                   12
glucose_change_10min                 12
glucose_prev_5min                     6
glucose_rate_5min                     6
glucose_change_5min                   6
glucose_rolling_std_30min             6
sleep_change                          6


In [36]:
print("Columns with missing values:", len(missing))

Columns with missing values: 25


In [37]:
# Calculate medians ONLY from training data
train_medians = X_train.median(numeric_only=True)

# Fill numeric missing values
X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)

print("Remaining NaNs in X_train:", X_train.isna().sum().sum())
print("Remaining NaNs in X_test :", X_test.isna().sum().sum())

Remaining NaNs in X_train: 0
Remaining NaNs in X_test : 0


In [38]:
print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

print("NaN in X_train:", X_train.isna().sum().sum())
print("NaN in X_test :", X_test.isna().sum().sum())

X_train shape: (51424, 59)
X_test shape : (12859, 59)
NaN in X_train: 0
NaN in X_test : 0


In [39]:
missing = X_train.isna().sum()

missing = missing[missing > 0].sort_values(ascending=False)

print(missing)

Series([], dtype: int64)


In [40]:
# Calculate medians ONLY from training data
train_medians = X_train.median(numeric_only=True)

# Fill numeric missing values
X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)

print("Remaining NaNs in X_train:", X_train.isna().sum().sum())
print("Remaining NaNs in X_test :", X_test.isna().sum().sum())

Remaining NaNs in X_train: 0
Remaining NaNs in X_test : 0


In [41]:
print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

print("NaN in X_train:", X_train.isna().sum().sum())
print("NaN in X_test :", X_test.isna().sum().sum())

X_train shape: (51424, 59)
X_test shape : (12859, 59)
NaN in X_train: 0
NaN in X_test : 0


In [42]:
X_train_np = X_train.astype("float32").values
X_test_np = X_test.astype("float32").values

y_train_np = y_train.astype("float32").values
y_test_np = y_test.astype("float32").values

print("Any NaN:", np.isnan(X_train_np).any())
print("Any Inf:", np.isinf(X_train_np).any())

Any NaN: False
Any Inf: False


In [43]:
print(missing)

Series([], dtype: int64)


In [44]:
print("NaN in X_train:", X_train.isna().sum().sum())
print("NaN in X_test :", X_test.isna().sum().sum())

NaN in X_train: 0
NaN in X_test : 0
